___
# Notebook W4.1

In [1]:
import pandas as pd
import hvplot.pandas
from ipywidgets import (
    Dropdown,
    Button,
    Layout,
    Output,
    GridspecLayout,
    interact,
)

from utils import get_cols_for_display, apply_filter_mask, make_venn_diagram, find_outliers

pd.set_option('display.max_colwidth', None)
button_color = 'lightblue'

## Load data

In [2]:
dataset = pd.read_parquet("../data/t2dm_synthetic_T2DM_only_prepared.parquet", engine="pyarrow")
dataset

,patient_id,hospital,age,sex,height,weight,weight_over_100,bmi,waist,obesity,...,egfr,islet_antibody,coeliac_antibody,thyroid_function,thyroid_antibody,vitamin_b12,dka_diagnosis,admissions_in_2024,visits_before_2025,outcome_admission_in_2025
0,922904,WH,18,F,173.0,107.9,True,36.486302,91.9,False,...,NaN,True,False,True,False,False,False,0,2,False
1,118725,RMH,19,M,158.0,89.9,False,36.192468,84.5,False,...,NaN,False,False,False,False,False,False,0,3,False
2,888078,RMH,19,F,169.0,134.9,True,47.395225,105.7,False,...,NaN,False,False,False,False,False,False,12,14,False
3,811787,WH,20,M,167.0,NaN,False,NaN,NaN,False,...,NaN,True,True,True,True,False,False,0,2,False
4,334671,WH,28,F,162.0,NaN,False,NaN,NaN,False,...,NaN,True,False,False,False,False,False,0,12,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3822,695216,WH,86,M,180.0,77.2,False,24.209071,85.0,False,...,NaN,False,False,False,False,False,False,2,2,False
3823,694586,WH,87,F,169.0,NaN,False,NaN,NaN,False,...,NaN,False,False,False,False,False,False,8,13,True
3824,161717,WH,86,M,167.0,96.5,False,36.069186,82.8,False,...,NaN,False,False,False,False,False,False,1,14,True
3825,855731,WH,86,F,162.0,78.3,False,30.028990,73.4,False,...,NaN,False,False,True,False,True,False,0,49,False


### Skim the dataset

In [3]:
dataset.head(15)

,patient_id,hospital,age,sex,height,weight,weight_over_100,bmi,waist,obesity,...,egfr,islet_antibody,coeliac_antibody,thyroid_function,thyroid_antibody,vitamin_b12,dka_diagnosis,admissions_in_2024,visits_before_2025,outcome_admission_in_2025
0,922904,WH,18,F,173.0,107.9,True,36.486302,91.9,False,...,NaN,True,False,True,False,False,False,0,2,False
1,118725,RMH,19,M,158.0,89.9,False,36.192468,84.5,False,...,NaN,False,False,False,False,False,False,0,3,False
2,888078,RMH,19,F,169.0,134.9,True,47.395225,105.7,False,...,NaN,False,False,False,False,False,False,12,14,False
3,811787,WH,20,M,167.0,NaN,False,NaN,NaN,False,...,NaN,True,True,True,True,False,False,0,2,False
4,334671,WH,28,F,162.0,NaN,False,NaN,NaN,False,...,NaN,True,False,False,False,False,False,0,12,True
5,407486,WH,24,F,145.0,85.9,False,40.973278,84.9,False,...,NaN,False,False,False,False,False,False,0,1,False
6,876505,WH,29,F,161.0,NaN,False,NaN,NaN,False,...,NaN,True,False,False,False,False,False,0,10,True
7,188919,RMH,30,F,159.0,119.2,True,48.377043,91.8,False,...,NaN,False,False,False,False,False,False,0,38,True
8,231150,RMH,26,F,169.0,88.8,False,31.387520,82.4,False,...,NaN,False,False,False,False,False,False,0,8,False
9,887565,WH,26,F,164.0,85.9,False,32.347511,80.8,False,...,NaN,False,False,False,False,False,False,0,2,False


## Examine column types

In [4]:
# Map column names to types and descriptions for display
cols_for_display = get_cols_for_display(dataset)

In [5]:
# Interactive widget to select a type of feature and display the corresponding columns
def retrieve_cols_for_display(col_type):
    if col_type != 'All':
        return cols_for_display[(cols_for_display['type'] == col_type)]
    else:
        return cols_for_display

data_types_list = Dropdown(
    options=['Discrete', 'Continuous', 'Boolean', 'Categorical', 'All'],
    description='Select a type of feature:',
    disabled=False,
    style={'description_width': 'max-content'},
    value='Discrete'
)

interact(retrieve_cols_for_display, col_type=data_types_list);

interactive(children=(Dropdown(description='Select a type of feature:', options=('Discrete', 'Continuous', 'Bo…

## Subset records

In [6]:
# Operator symbol mapping: display label → Python operator string
operator_dict = {"=": "==", "≥": ">=", ">": ">", "!=": "!=", "<": "<", "≤": "<="}
# Reverse mapping: Python operator string → display label (used for Venn labels)
r_operator_dict = dict(zip(operator_dict.values(), operator_dict.keys()))

# --- Widget definitions ---
# Left operand widgets
widget_dataset = Dropdown(description='Table:', options=['dataset'])
widget_col1 = Dropdown(description='Column 1:', options=dataset.columns, value='sex')
widget_col1_op = Dropdown(description='Operator:', options=operator_dict)
widget_col1_value = Dropdown(description='Value:', options=sorted(dataset[widget_col1.value].unique()))
# Right operand widgets
widget_col2 = Dropdown(description='Column 2:', options=dataset.columns, value='method_manage_t2dm')
widget_col2_op = Dropdown(description='Operator:', options=operator_dict)
widget_col2_value = Dropdown(description='Value:', options=sorted(dataset[widget_col2.value].unique()))
# Boolean logic widget
boolean_logic = Dropdown(description='Boolean condition:', options=['AND (∩)', 'OR (∪)'], style={'description_width': 'max-content'})
# Action button
button1 = Button(description='Show result', layout=Layout(width='100%'))
button1.style.button_color = button_color

# --- Event handlers: refresh value dropdown when column selection changes ---
def update_value_options(val_dropdown, change):
    val_dropdown.options = sorted(dataset[change.new].unique())

widget_col1.observe(lambda change: update_value_options(widget_col1_value, change), names=['value'])
widget_col2.observe(lambda change: update_value_options(widget_col2_value, change), names=['value'])

output1 = Output()

# --- Button handler: apply filters, plot Venn diagram, display results ---
def on_button_clicked1(_):
    pd.set_option('display.max_rows', 50)
    output1.clear_output()

    if widget_col1_value.value is None or widget_col2_value.value is None:
        return

    # Build boolean masks using operator functions (avoids eval)
    mask1 = apply_filter_mask(dataset, widget_col1.value, widget_col1_op.value, widget_col1_value.value)
    mask2 = apply_filter_mask(dataset, widget_col2.value, widget_col2_op.value, widget_col2_value.value)

    is_and = boolean_logic.value == 'AND (∩)'
    combined_mask = mask1 & mask2 if is_and else mask1 | mask2
    result_dataset = dataset[combined_mask]

    # Patient keys matching each individual condition and their intersection
    subs1 = dataset.loc[mask1, 'patient_id'].tolist()
    subs2 = dataset.loc[mask2, 'patient_id'].tolist()
    inter = list(set(subs1) & set(subs2))

    # Labels for Venn diagram sets
    label1 = f"{widget_col1.value} {r_operator_dict[widget_col1_op.value]} {widget_col1_value.value}"
    label2 = f"{widget_col2.value} {r_operator_dict[widget_col2_op.value]} {widget_col2_value.value}"

    with output1:
        if is_and:
            make_venn_diagram(subs1, subs2, inter, label1, label2, colors=('purple', 'skyblue'))
            print('Number of rows in result:', len(inter))
        else:
            make_venn_diagram(subs1, subs2, inter, label1, label2, colors=('skyblue', 'skyblue'))
            print('Number of rows in result:', result_dataset.shape[0])
        display(result_dataset)

button1.on_click(on_button_clicked1)

# --- Grid layout (3 rows × 3 cols) ---
grid1 = GridspecLayout(3, 3)
grid1[0, 0] = widget_col1
grid1[1, 0] = widget_col1_op
grid1[2, 0] = widget_col1_value
grid1[0, 1] = boolean_logic
grid1[0, 2] = widget_col2
grid1[1, 2] = widget_col2_op
grid1[2, 2] = widget_col2_value
grid1[2, 1] = button1

display(grid1)
display(output1)

GridspecLayout(children=(Dropdown(description='Column 1:', index=3, layout=Layout(grid_area='widget001'), opti…

Output()

## Descriptive statistics

In [7]:
button2 = Button(description ='Generate descriptive statistics for discrete and continuous variables', layout=Layout(width='50%'))
button2.style.button_color = button_color
output2 = Output()

def on_button_clicked2(_):
    output2.clear_output()
    with output2:
        display(dataset.drop('patient_id', axis=1).describe(include=['int64', 'float64']).T.head(50))

button2.on_click(on_button_clicked2)

display(button2)
display(output2)

Button(description='Generate descriptive statistics for discrete and continuous variables', layout=Layout(widt…

Output()

In [8]:
button3 = Button(description ='Generate descriptive statistics for categorical and boolean variables', layout=Layout(width='50%'))
button3.style.button_color = button_color
output3 = Output()

def on_button_clicked3(_):
    output3.clear_output()
    with output3:
        display(dataset.describe(include=['category', 'bool']).T.head(50))

button3.on_click(on_button_clicked3)

display(button3)
display(output3)

Button(description='Generate descriptive statistics for categorical and boolean variables', layout=Layout(widt…

Output()

## Missing values

In [9]:
missing_values = dataset.isna().sum().sort_values(ascending=False)
missing_values = missing_values[missing_values > 0]
missing_values = missing_values.to_frame(name='Number of missing values')

In [10]:
button4 = Button(description ='Show number of missing values', layout=Layout(width='50%'))
button4.style.button_color = button_color
output4 = Output()

def on_button_clicked4(_):
    output4.clear_output()
    n_values = dataset.shape[0]
    missing_values = dataset.isna().sum().sort_values(ascending=False)
    missing_values = missing_values[missing_values > 0]
    missing_values = missing_values.to_frame(name='Number of missing values')
    missing_values['Proportion of missing values'] = (missing_values['Number of missing values'] / n_values * 100).round(2)
    with output4:
        display(missing_values)

button4.on_click(on_button_clicked4)

display(button4)
display(output4)

Button(description='Show number of missing values', layout=Layout(width='50%'), style=ButtonStyle(button_color…

Output()

## Examine outliers

In [11]:
num_vars_list = Dropdown(
    options=dataset.drop('patient_id', axis=1).select_dtypes(include=['int64', 'float64']).columns, 
    description='Select a column to view:', 
    disabled=False,
    style={'description_width': 'max-content'}, 
    value = 'years_diagnosed'
    )

def retrieve_col_for_display(col):
    outliers = find_outliers(dataset[col])
    n_outliers = len(outliers)
    print('Column "' + col + '" plotted with outliers')
    print('Number of outliers:', n_outliers)
    return dataset.hvplot.box(col,  width=450) + dataset.hvplot.hist(col,  width=450)

interact(retrieve_col_for_display, col=num_vars_list);

interactive(children=(Dropdown(description='Select a column to view:', index=7, options=('age', 'height', 'wei…

### Filter outliers

In [ ]:
# cols_list = Dropdown(
#     options=dataset.drop('patient_id', axis=1).select_dtypes(include=['int64', 'float64']).columns, 
#     description='Select a column to view:', 
#     disabled=False,
#     style={'description_width': 'max-content'}, 
#     value = 'years_diagnosed'
#     )
button5 = Button(description ='Filter outliers', layout=Layout(width='50%'))
button5.style.button_color = button_color
output5 = Output()

def on_button_clicked5(_):
    output5.clear_output()
    outliers = find_outliers(dataset[num_vars_list.value])
    n_outliers = len(outliers)
    dataset_new = dataset[~dataset[num_vars_list.value].isin(outliers)]
    with output5:
        print('Column "' + num_vars_list.value + '" plotted without outliers')
        print('Number of outliers removed:', n_outliers)
        display(dataset_new.hvplot.box(num_vars_list.value,  width=450, kwargs={'showfliers': False}) + dataset_new.hvplot.hist(num_vars_list.value,  width=450))

button5.on_click(on_button_clicked5)
display(button5)
display(output5)

Button(description='Filter outliers', layout=Layout(width='50%'), style=ButtonStyle(button_color='lightblue'))

Output()

## Convert categorical variables to numbers

In [17]:
# Interactive widget to select a categorical variable and display the value counts
cat_vars_list = Dropdown(
    options = dataset.select_dtypes(include='category'),
    description ='Categorical features:',
    style={'description_width': 'max-content'}, 
    value = 'hospital')

def retrieve_value_counts(col):
    return dataset[col].value_counts().to_frame(name='Number of values')

interact(retrieve_value_counts, col=cat_vars_list);

interactive(children=(Dropdown(description='Categorical features:', options=('hospital', 'sex', 'smoker', 'met…

In [21]:
button6 = Button(description ='Simple encoding of a categorical variable',layout=Layout(width='50%'))
button6.style.button_color = button_color
output6 = Output()

def on_button_clicked6(_):
    output6.clear_output()
    mapping  = dict(enumerate(dataset[cat_vars_list.value].cat.categories))
    with output6:
        print("Mapping categories to numbers:")
        for code, category in mapping.items():
            print(f"{category} is mapped to {code}")
        print()
        display(pd.concat([
            dataset.patient_id,
            dataset[cat_vars_list.value],
            pd.Series(dataset[cat_vars_list.value].cat.codes, name=cat_vars_list.value)
            ], axis=1).head(10))

button6.on_click(on_button_clicked6)
display(button6)
display(output6)

Button(description='Simple encoding of a categorical variable', layout=Layout(width='50%'), style=ButtonStyle(…

Output()

In [20]:
button7 = Button(description ='One-hot encoding of a categorical variable',layout=Layout(width='50%'))
button7.style.button_color = button_color
output7 = Output()

def on_button_clicked7(_):
    output7.clear_output()
    with output7:
        display(pd.concat([
            dataset.patient_id,
            dataset[cat_vars_list.value],
            pd.get_dummies(dataset[cat_vars_list.value])
            ], axis=1).head(10))

button7.on_click(on_button_clicked7)
display(button7)
display(output7)

Button(description='One-hot encoding of a categorical variable', layout=Layout(width='50%'), style=ButtonStyle…

Output()

## Smart imputation